# Simple RAG 2 (Multiple Documents) - Document Chunking and Embedding

**Flow:**

Documents → Chunk → Embed → Store (ChromaDB)

## Install Dependencies

In [2]:
# Conda environment setup
#!pip install langchain langchain-chroma langchain-openai chromadb pypdf

## Basic RAG Code Phase 1 - Document Ingestion and Embedding

In [3]:
# # Colab setup
# from google.colab import drive
# from google.colab import userdata
# drive.mount('/content/drive')

In [4]:
# # Colab File Location Setup
# file_location = "/content/drive/MyDrive/rag_langchain/data/rag_sample1.txt"
# store_location = "/content/drive/MyDrive/rag_langchain/data/chroma_db1"

In [5]:
# # Colab Key
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [1]:
# Local File Location Setup
data_folder = "../data"
store_location = "../vector_db/chroma_db2"

In [2]:
# Use Python dotenv to load environment variables from a .env file
from dotenv import load_dotenv
import os

load_dotenv()


True

### Step 1: Load & split documents

In [4]:
from pathlib import Path

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_documents(data_dir: str) -> list[Document]:
    docs = []
    for file_path in sorted(Path(data_dir).rglob("*")):
        if not file_path.is_file():
            continue

        suffix = file_path.suffix.lower()
        if suffix == ".txt":
            text = file_path.read_text(encoding="utf-8")
            docs.append(
                Document(
                    page_content=text,
                    metadata={"source": str(file_path)}
                )
            )
        elif suffix == ".pdf":
            reader = PdfReader(str(file_path))
            for page_number, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                if text:
                    docs.append(
                        Document(
                            page_content=text,
                            metadata={"source": str(file_path), "page": page_number}
                        )
                    )

    return docs

docs = load_documents(data_folder)
if not docs:
    raise ValueError(f"No supported documents found in {data_folder}")

print(f"Loaded {len(docs)} document/page item(s)")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

splits = splitter.split_documents(docs)
print(f"Created {len(splits)} chunks")

Loaded 193 document/page item(s)
Created 3154 chunks


### Step 2: Create embeddings + store in Chroma

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=store_location
)